# AIMLC ZG521 — Conversational AI · Group Assignment 1
## Problem Statement 2 — Study of Embedding Models and Approximate Nearest Neighbor Search: Semantic Quality vs Search Efficiency

**Group 129** · Total: 10 Marks · Deadline: 28 Aug 2026

## Student Details

| Name | BITS ID | Email |
|---|---|---|
| *TODO — fill in* | | |
| *TODO — fill in* | | |
| *TODO — fill in* | | |
| *TODO — fill in* | | |

## Contribution by Each Student

| Member | Area | Task(s) | Section(s) done |
|---|---|---|---|
| P1 | Dataset & embedding + Visualization & reporting | T1, T2, T8, report assembly | *TODO — name* |
| P2 | Similarity matrix comparison + kNN baseline | T3, T4 | *TODO — name* |
| P3 | HNSW vs IVF | T5 | *TODO — name* |
| P4 | Evaluation & analysis | T6, T7 | *TODO — name* |

(Per `ass-1/ROADMAP.md` day-by-day execution plan.)

## Problem Statement

Study of embedding models and approximate nearest neighbor (ANN) search, comparing **semantic quality vs search efficiency**:
- **Module 1** — Dataset and embedding preparation (1 mark)
- **Module 2** — Similarity metrics and exact retrieval (3 marks)
- **Module 3** — ANN search experiment: HNSW vs IVF (3 marks)
- **Module 4** — Embedding quality analysis and final recommendation (3 marks)

Full task-by-task plan: `ass-1/ROADMAP.md`.

## Tools and Libraries Used

- **Python 3.10**
- `datasets` (Hugging Face) — loading the BEIR-format retrieval dataset
- `pandas`, `numpy` — data handling
- `sentence-transformers` — encoder embedding models (Task 2 onward)
- `faiss-cpu` — HNSW / IVF ANN indexes (Task 5 onward)
- `matplotlib` — plots (Task 6 onward)
- Environment: course-provided remote system (see `ass-1/ROADMAP.md` — "Resolved clarifications")

In [4]:
import sys, time, json, random
import numpy as np
import pandas as pd

random.seed(129)   # Group 129 — fixed seed for reproducibility
np.random.seed(129)

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy :", np.__version__)

Python: 3.11.5
pandas: 3.0.5
numpy : 2.4.6


---
# Task 1 — Corpus and Query Dataset Preparation (0.5 Marks)

**Dataset chosen: `SciFact`** (from the BEIR benchmark — Thakur et al., 2021), via its scientific-claim-verification source (Wadden et al., 2020).

**Why SciFact:** it is a publicly available, BEIR-format dataset that ships a document corpus, a query set, and **query-document relevance labels (qrels)** together, in exactly the shape this task asks for — no relevance judgments need to be hand-built. Its corpus (~5,183 passages) and query set (~300 claims) both comfortably clear the assignment's 1,000-passage / 50-query minimums.

In [5]:
from datasets import load_dataset

def load_beir_scifact():
    '''
    Load the SciFact BEIR dataset (corpus, queries, qrels) from Hugging Face.

    This is the standard, correct way to load a BEIR-format dataset and is what
    should run on the course-provided remote system (which has full internet
    access). If the network path to huggingface.co is blocked (as it is inside
    this authoring sandbox), we fall back to a small local demo sample so the
    rest of the pipeline below can still be run and verified end to end.
    Re-run this cell on the remote system to pull the real ~5K/300 SciFact data.
    '''
    try:
        corpus_ds = load_dataset("BeIR/scifact", "corpus", split="corpus")
        queries_ds = load_dataset("BeIR/scifact", "queries", split="queries")
        qrels_ds = load_dataset("BeIR/scifact-qrels", split="test")

        corpus_df = corpus_ds.to_pandas().rename(columns={"_id": "doc_id"})
        queries_df = queries_ds.to_pandas().rename(columns={"_id": "query_id"})
        qrels_df = qrels_ds.to_pandas()
        qrels_df.columns = ["query_id", "doc_id", "relevance"]
        # BeIR-qrels ids are stored as ints in this mirror; corpus/query ids are strings
        qrels_df["query_id"] = qrels_df["query_id"].astype(str)
        qrels_df["doc_id"] = qrels_df["doc_id"].astype(str)

        return corpus_df, queries_df, qrels_df, "live-huggingface"
    except Exception as e:
        print(f"[warning] Live download failed in this environment: {e!r}")
        print("[warning] Falling back to a small local DEMO sample (NOT the final dataset).")
        return _local_demo_sample() + ("local-demo-fallback",)


def _local_demo_sample():
    '''
    Tiny, clearly-labelled stand-in used ONLY when the live download is unreachable
    (as in this authoring sandbox, which allows outbound access to pypi.org only).
    This sample is deliberately far below the 1,000-passage / 50-query minimums --
    it exists purely to prove the validation/processing code below is correct, not
    to serve as the submitted dataset.
    '''
    demo_corpus = [
        {"doc_id": f"D{i}", "title": f"Demo abstract {i}",
         "text": t}
        for i, t in enumerate([
            "Regular exercise improves cardiovascular health and reduces resting heart rate.",
            "Vitamin D deficiency is associated with increased risk of bone fractures.",
            "Machine learning models can predict protein folding structures from sequence data.",
            "Antibiotic resistance in bacteria develops through horizontal gene transfer.",
            "Climate change is causing measurable shifts in global precipitation patterns.",
            "mRNA vaccines train the immune system by encoding a viral antigen.",
            "Sleep deprivation impairs memory consolidation in the hippocampus.",
            "Coral reefs are highly sensitive to small increases in ocean temperature.",
            "Gut microbiota composition influences host metabolism and immune response.",
            "Solar panel efficiency has improved significantly with perovskite materials.",
        ])
    ]
    demo_queries = [
        {"query_id": "Q1", "text": "Does exercise help the heart?"},
        {"query_id": "Q2", "text": "What happens when bacteria resist antibiotics?"},
    ]
    demo_qrels = [
        {"query_id": "Q1", "doc_id": "D0", "relevance": 1},
        {"query_id": "Q2", "doc_id": "D3", "relevance": 1},
    ]
    return pd.DataFrame(demo_corpus), pd.DataFrame(demo_queries), pd.DataFrame(demo_qrels)


corpus_df, queries_df, qrels_df, data_source = load_beir_scifact()
print(f"\ndata_source = {data_source!r}")
print(f"corpus  : {len(corpus_df)} passages")
print(f"queries : {len(queries_df)} queries")
print(f"qrels   : {len(qrels_df)} relevance judgments")

/Users/sanjaykumarpushadapu/Projects/MTech/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test split: 100%|██████████| 339/339 [00:00<00:00, 138759.54 examples/s]


data_source = 'live-huggingface'
corpus  : 5183 passages
queries : 1109 queries
qrels   : 339 relevance judgments


In [6]:
# Validate against the assignment's stated minimums
MIN_CORPUS, MIN_QUERIES = 1000, 50

meets_corpus_min = len(corpus_df) >= MIN_CORPUS
meets_query_min = len(queries_df) >= MIN_QUERIES

print(f"Corpus  >= {MIN_CORPUS}: {meets_corpus_min}  ({len(corpus_df)} passages)")
print(f"Queries >= {MIN_QUERIES}: {meets_query_min}  ({len(queries_df)} queries)")
print(f"Every query has >=1 relevance judgment: "
      f"{queries_df['query_id'].isin(qrels_df['query_id']).all() if len(queries_df) else 'n/a'}")

if data_source == "live-huggingface":
    assert meets_corpus_min, "Corpus below the 1,000-passage minimum"
    assert meets_query_min, "Query set below the 50-query minimum"
    print("\n[OK] Live SciFact data clears both minimums.")
else:
    print("\n[NOTE] Running on the local DEMO fallback -- minimums are NOT expected to "
          "pass here. Re-run this notebook on the remote system with internet access "
          "to pull the real SciFact data (~5,183 passages / ~300 queries), which clears "
          "both minimums comfortably.")

Corpus  >= 1000: True  (5183 passages)
Queries >= 50: True  (1109 queries)
Every query has >=1 relevance judgment: False

[OK] Live SciFact data clears both minimums.


In [7]:
# Inspect a sample of each piece
print("=== Sample corpus passage ===")
print(corpus_df.iloc[0].to_dict())

print("\n=== Sample query ===")
print(queries_df.iloc[0].to_dict())

print("\n=== Sample relevance judgments (qrels) ===")
print(qrels_df.head())

=== Sample corpus passage ===
{'doc_id': '4983', 'title': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.', 'text': 'Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior 

In [8]:
# Persist to disk so Tasks 2+ (embedding generation, retrieval, ANN search)
# can load the same corpus/queries/qrels without re-running this cell.
import os
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

corpus_df.to_json(f"{DATA_DIR}/corpus.jsonl", orient="records", lines=True)
queries_df.to_json(f"{DATA_DIR}/queries.jsonl", orient="records", lines=True)
qrels_df.to_csv(f"{DATA_DIR}/qrels.tsv", sep="\t", index=False)

print(f"Saved to {DATA_DIR}/: corpus.jsonl, queries.jsonl, qrels.tsv (source={data_source})")

Saved to data/: corpus.jsonl, queries.jsonl, qrels.tsv (source=live-huggingface)


### Dataset Details and Source

- **Name:** SciFact (from the BEIR benchmark)
- **Domain:** scientific-claim verification abstracts (biomedical / life sciences)
- **Source:** Wadden et al., *"Fact or Fiction: Verifying Scientific Claims"*, EMNLP 2020; redistributed in BEIR format by Thakur et al., *"BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of Information Retrieval Models"*, NeurIPS 2021 (Datasets & Benchmarks track)
- **Access:** `BeIR/scifact` (corpus, queries) and `BeIR/scifact-qrels` (relevance judgments) on the Hugging Face Hub
- **Size (full dataset):** ~5,183 corpus passages, ~300 queries (test split), with binary relevance judgments (qrels) linking each query to its relevant document(s)
- **Format:** corpus = `{doc_id, title, text}`; queries = `{query_id, text}`; qrels = `{query_id, doc_id, relevance}`

### Explanation of the Logic Used

`load_beir_scifact()` pulls the three BEIR components directly from the Hugging Face Hub via `datasets.load_dataset`, normalizes column names to a consistent `doc_id` / `query_id` schema across all three tables, and casts id columns to strings so joins between queries/corpus/qrels are reliable. A fallback path (`_local_demo_sample`) exists purely so the validation and inspection cells below can still execute in a network-restricted environment; it is clearly logged and is never mistaken for the real submission data because `data_source` is printed and checked before any minimum-size assertion runs.

### Justification for the Chosen Approach

BEIR-format datasets bundle corpus, queries, *and* relevance labels in one consistent schema, which the assignment explicitly permits ("students may use an existing dataset containing query-document relevance labels") and which removes an entire class of errors (inconsistent manual relevance judgments) that a self-built dataset would risk. SciFact specifically was chosen over larger BEIR datasets (e.g. FiQA-2018) because its domain (scientific claims) gives clean, unambiguous relevance judgments — useful later in Task 3 and Task 7 when similarity rankings and qualitative differences need to be explained against ground truth that isn't itself noisy.

### Inference

Once run with internet access, the loaded corpus and query set both exceed the assignment's minimums by a wide margin (~5x the corpus minimum, ~6x the query minimum), which leaves comfortable room to subsample later if compute time becomes a constraint in Tasks 2/5 without ever dropping below 1,000 passages or 50 queries.

### Limitations Observed

- SciFact's relevance judgments are binary (relevant / not relevant) with no graded relevance — fine for Recall@5 in Tasks 4–6, but it means Task 3's "which metric appears most suitable" analysis has less ranking nuance to work with than a graded-relevance dataset would give.
- This sandbox's outbound network is restricted to `pypi.org`; the live Hugging Face download path could not be executed or verified here and must be re-run on the course-provided remote system.

### Possible Improvements

- Cross-check a second BEIR dataset (e.g. NFCorpus) as a robustness check on Task 7's qualitative findings, time permitting.
- If Task 5/6 index-building is slow at full scale, subsample the corpus in a stratified way (keeping every relevant document for every query) rather than a pure random subsample.

### References

- Thakur, N. et al. (2021). *BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of Information Retrieval Models.* NeurIPS Datasets & Benchmarks.
- Wadden, D. et al. (2020). *Fact or Fiction: Verifying Scientific Claims.* EMNLP.
- Dataset card: `https://huggingface.co/datasets/BeIR/scifact`

---
# Task 2 — Embedding Generation and Pooling (0.5 Marks)
*Owner: P1 — see `ass-1/ROADMAP.md`, Days 1–2. Not started.*

---
# Task 3 — Similarity Metric Comparison (1.5 Marks)
*Owner: P2 — see `ass-1/ROADMAP.md`, Days 3–4. Not started.*

---
# Task 4 — Exact kNN Baseline (1.5 Marks)
*Owner: P2 — see `ass-1/ROADMAP.md`, Days 3–4. Not started.*

---
# Task 5 — HNSW vs IVF (2 Marks)
*Owner: P3 — see `ass-1/ROADMAP.md`, Days 3–6. Not started.*

---
# Task 6 — ANN Trade-off Analysis (1 Mark)
*Owner: P4 — see `ass-1/ROADMAP.md`, Day 7. Not started.*

---
# Task 7 — Qualitative Retrieval Analysis (2 Marks)
*Owner: P4 — see `ass-1/ROADMAP.md`, Days 3–4. Not started.*

---
# Task 8 — Final Recommendation (1 Mark)
*Owners: P1 + P4 — see `ass-1/ROADMAP.md`, Day 8. Not started.*

---
# Final Conclusion
*To be written once Tasks 2–8 are complete — must cover key observations, strengths, limitations, and possible future improvements (see `ass-1/ROADMAP.md` grading risk flags, #16).*

# References
*Consolidated reference list — add each task's citations here as they're completed.*

- Thakur, N. et al. (2021). BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of Information Retrieval Models. NeurIPS.
- Wadden, D. et al. (2020). Fact or Fiction: Verifying Scientific Claims. EMNLP.